This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch --upgrade -q

## Introduction to PyTorch

### A brief history of deep learning frameworks

### How these frameworks relate to each other

#### First steps with PyTorch

##### Tensors and parameters in PyTorch

###### Constant tensors

In [0]:
import torch
torch.ones(size=(2, 1))

In [0]:
torch.zeros(size=(2, 1))

In [0]:
torch.tensor([1, 2, 3], dtype=torch.float32)

###### Random tensors

In [0]:
torch.normal(
mean=torch.zeros(size=(3, 1)),
std=torch.ones(size=(3, 1)))

In [0]:
torch.rand(3, 1)

###### Tensor assignment and the Parameter class

In [0]:
x = torch.zeros(size=(2, 1))
x[0, 0] = 1.
x

In [0]:
x = torch.zeros(size=(2, 1))
p = torch.nn.parameter.Parameter(data=x)

##### Tensor operations: Doing math in PyTorch

In [0]:
a = torch.ones((2, 2))
b = torch.square(a)
c = torch.sqrt(a)
d = b + c
e = torch.matmul(a, b)
f = torch.cat((a, b), dim=0)

In [0]:
def dense(inputs, W, b):
    return torch.relu(torch.matmul(inputs, W) + b)

##### Computing gradients with PyTorch

In [0]:
input_var = torch.tensor(3.0, requires_grad=True)
result = torch.square(input_var)
result.backward()
gradient = input_var.grad
gradient

In [0]:
result = torch.square(input_var)
result.backward()
input_var.grad

In [0]:
input_var.grad = None

#### An end-to-end example: A linear classifier in pure PyTorch

In [0]:
import numpy as np

num_samples_per_class = 1000
negative_samples = np.random.multivariate_normal(
    mean=[0, 3], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)
positive_samples = np.random.multivariate_normal(
    mean=[3, 0], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)

In [0]:
inputs = np.vstack((negative_samples, positive_samples)).astype(np.float32)

In [0]:
targets = np.vstack(
    (
        np.zeros((num_samples_per_class, 1), dtype="float32"),
        np.ones((num_samples_per_class, 1), dtype="float32"),
    )
)

In [0]:
import matplotlib.pyplot as plt

plt.scatter(inputs[:, 0], inputs[:, 1], c=targets[:, 0])
plt.show()

In [0]:
input_dim = 2
output_dim = 1

W = torch.rand(input_dim, output_dim, requires_grad=True)
b = torch.zeros(output_dim, requires_grad=True)

In [0]:
def model(inputs, W, b):
    return torch.matmul(inputs, W) + b

In [0]:
def mean_squared_error(targets, predictions):
    per_sample_losses = torch.square(targets - predictions)
    return torch.mean(per_sample_losses)

In [ ]:
learning_rate = 0.1

def training_step(inputs, targets, W, b):
    # PyTorch: pass W and b explicitly (the original call dropped them).
    predictions = model(inputs, W, b)
    loss = mean_squared_error(targets, predictions)
    loss.backward()
    grad_loss_wrt_W, grad_loss_wrt_b = W.grad, b.grad
    with torch.no_grad():
        W -= grad_loss_wrt_W * learning_rate
        b -= grad_loss_wrt_b * learning_rate
    W.grad = None
    b.grad = None
    return loss

In [ ]:
# PyTorch: drive the from-scratch training loop, feeding tensors built from
# the numpy `inputs`/`targets` arrays.
torch_inputs = torch.from_numpy(inputs)
torch_targets = torch.from_numpy(targets)
for step in range(40):
    loss = training_step(torch_inputs, torch_targets, W, b)
    print(f"Loss at step {step}: {loss:.4f}")

In [ ]:
with torch.no_grad():
    predictions = model(torch_inputs, W, b).numpy()
plt.scatter(inputs[:, 0], inputs[:, 1], c=predictions[:, 0] > 0.5)
plt.show()

In [ ]:
# PyTorch: detach the learned weights to numpy before plotting the boundary.
W_np, b_np = W.detach().numpy(), b.detach().numpy()
x = np.linspace(-1, 4, 100)
y = -W_np[0] / W_np[1] * x + (0.5 - b_np) / W_np[1]
plt.plot(x, y, "-r")
plt.scatter(inputs[:, 0], inputs[:, 1], c=predictions[:, 0] > 0.5)

##### Packaging state and computation with the Module class

In [0]:
class LinearModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.W = torch.nn.Parameter(torch.rand(input_dim, output_dim))
        self.b = torch.nn.Parameter(torch.zeros(output_dim))

    def forward(self, inputs):
        return torch.matmul(inputs, self.W) + self.b

In [0]:
model = LinearModel()

In [0]:
torch_inputs = torch.tensor(inputs)
output = model(torch_inputs)

In [0]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [0]:
def training_step(inputs, targets):
    predictions = model(inputs)
    loss = mean_squared_error(targets, predictions)
    loss.backward()
    optimizer.step()
    model.zero_grad()
    return loss

##### Making PyTorch modules fast using compilation

In [0]:
compiled_model = torch.compile(model)

In [0]:
@torch.compile
def dense(inputs, W, b):
    return torch.relu(torch.matmul(inputs, W) + b)

#### What makes the PyTorch approach unique

### High-level model building with torch.nn

#### First steps with torch.nn

#### Layers: The building blocks of deep learning

##### The base `Module` class in PyTorch

In [ ]:
from torch import nn

# PyTorch: a custom layer subclasses nn.Module. There's no lazy build(), so we
# take input_dim up front and register weights as nn.Parameter in __init__.
class SimpleDense(nn.Module):
    def __init__(self, input_dim, units, activation=None):
        super().__init__()
        self.units = units
        self.activation = activation
        self.W = nn.Parameter(torch.randn(input_dim, units) * 0.01)
        self.b = nn.Parameter(torch.zeros(units))

    def forward(self, inputs):
        y = torch.matmul(inputs, self.W) + self.b
        if self.activation is not None:
            y = self.activation(y)
        return y

In [ ]:
# PyTorch: instantiate with an explicit input_dim (784) instead of inferring it.
my_dense = SimpleDense(input_dim=784, units=32, activation=torch.relu)
input_tensor = torch.ones(size=(2, 784))
output_tensor = my_dense(input_tensor)
print(output_tensor.shape)

##### Composing layers into a model

In [ ]:
# PyTorch: nn.Linear is the built-in dense layer; activation is applied
# separately (e.g. via nn.ReLU or torch.relu).
layer = nn.Linear(in_features=784, out_features=32)

In [ ]:
# PyTorch: nn.Sequential needs explicit feature sizes and activation modules.
model = nn.Sequential(
    nn.Linear(784, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
)

In [ ]:
# PyTorch: stack our custom layers, threading input_dim through each one. We
# drop the final softmax activation and rely on nn.CrossEntropyLoss (which
# expects raw logits) at training time.
model = nn.Sequential(
    SimpleDense(784, 32, activation=torch.relu),
    SimpleDense(32, 64, activation=torch.relu),
    SimpleDense(64, 32, activation=torch.relu),
    SimpleDense(32, 10),
)

#### From layers to models

#### The "compile" step: Configuring the learning process

In [ ]:
# PyTorch: there is no compile() step. Build the model, then create the
# optimizer and loss function directly (rmsprop -> RMSprop, mse -> MSELoss).
model = nn.Sequential(nn.Linear(2, 1))
optimizer = torch.optim.RMSprop(model.parameters())
loss_fn = nn.MSELoss()

In [ ]:
# PyTorch: passing optimizer/loss objects is the only style here; "metrics"
# such as binary accuracy are computed by hand in the training/eval loop.
optimizer = torch.optim.RMSprop(model.parameters())
loss_fn = nn.MSELoss()

#### Picking a loss function

#### Writing the training loop

In [ ]:
# PyTorch: model.fit() becomes an explicit loop over shuffled mini-batches.
# We collect per-epoch loss into a history dict to mirror Keras's return value.
x = torch.from_numpy(inputs)
y = torch.from_numpy(targets)

batch_size = 128
history = {"loss": []}
for epoch in range(5):
    permutation = torch.randperm(len(x))
    epoch_losses = []
    for i in range(0, len(x), batch_size):
        idx = permutation[i : i + batch_size]
        predictions = model(x[idx])
        loss = loss_fn(predictions, y[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    history["loss"].append(sum(epoch_losses) / len(epoch_losses))
    print(f"Epoch {epoch}: loss {history['loss'][-1]:.4f}")

In [ ]:
history

#### Monitoring loss and metrics on validation data

In [ ]:
# PyTorch: recreate the model + optimizer, hold out a validation split, and add
# a per-epoch no-grad validation pass (loss + binary accuracy) by hand.
model = nn.Sequential(nn.Linear(2, 1))
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

indices_permutation = np.random.permutation(len(inputs))
shuffled_inputs = inputs[indices_permutation]
shuffled_targets = targets[indices_permutation]

num_validation_samples = int(0.3 * len(inputs))
val_inputs = shuffled_inputs[:num_validation_samples]
val_targets = shuffled_targets[:num_validation_samples]
training_inputs = shuffled_inputs[num_validation_samples:]
training_targets = shuffled_targets[num_validation_samples:]

x_train = torch.from_numpy(training_inputs)
y_train = torch.from_numpy(training_targets)
x_val = torch.from_numpy(val_inputs)
y_val = torch.from_numpy(val_targets)

batch_size = 16
for epoch in range(5):
    permutation = torch.randperm(len(x_train))
    for i in range(0, len(x_train), batch_size):
        idx = permutation[i : i + batch_size]
        loss = loss_fn(model(x_train[idx]), y_train[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        val_pred = model(x_val)
        val_loss = loss_fn(val_pred, y_val).item()
        val_acc = ((val_pred > 0.5).float() == y_val).float().mean().item()
    print(f"Epoch {epoch}: val_loss {val_loss:.4f} - val_accuracy {val_acc:.4f}")

#### Inference: Using a model after training

In [ ]:
# PyTorch: replace model.predict() with an eval/no-grad forward pass.
model.eval()
with torch.no_grad():
    predictions = model(torch.from_numpy(val_inputs)).numpy()
print(predictions[:10])